# 📝 Instrucciones: Construcción de Modelo de Regresión Lineal

## Contexto del Proyecto
**Datos socio demográficos y de recursos de salud a nivel de condado de EE. UU. (2018-2019)**

Se han recopilado datos socio demográficos y de recursos de salud por condado en los Estados Unidos. El objetivo es descubrir si existe alguna relación entre los recursos sanitarios y los datos socio demográficos.

> **Nota importante:** Para ello, es necesario que establezcas una **variable objetivo** (relacionada con la salud) para llevar a cabo el análisis.

---

### 📂 Paso 1: Carga del conjunto de datos
El conjunto de datos se encuentra en la carpeta del proyecto bajo el nombre `demographic_health_data.csv`. Puedes cargarlo directamente desde el siguiente enlace o añadirlo manualmente a tu repositorio:

* **URL del dataset:** [Descargar CSV](https://breathecode.herokuapp.com/asset/internal-link?id=733&path=demographic_health_data.csv)
* **Diccionario de variables:** Puedes consultar las definiciones [aquí](https://breathecode.herokuapp.com/asset/internal-link?id=733&path=data_dict.csv).

---

### 📊 Paso 2: Realiza un EDA completo
Este paso es vital para asegurar que nos quedamos con las variables estrictamente necesarias.
1.  **Limpieza:** Elimina variables no relevantes o que no aporten información.
2.  **Referencia:** Utiliza el Notebook de ejemplo y adáptalo a este caso de uso.
3.  **Split:** Divide el conjunto de datos en `train` y `test`.

---

### 🤖 Paso 3: Construye un modelo de regresión
Implementa y compara los siguientes modelos:

| Modelo | Descripción |
| :--- | :--- |
| **Regresión Lineal** | Implementación base y análisis de resultados iniciales. |
| **Modelo Lasso** | Construcción con atributos por defecto para comparar con la base. |

#### Análisis del Hiperparámetro
Analiza cómo evoluciona el coeficiente de determinación $R^2$ cuando el hiperparámetro del modelo Lasso cambia:
* **Rango de prueba:** Desde `0.0` hasta `20`.
* **Visualización:** Dibuja estos valores en un **diagrama de líneas**.



---

### ⚙️ Paso 4: Optimiza el modelo
Si los resultados del modelo Lasso no son satisfactorios tras el entrenamiento inicial, optimízalo empleando técnicas de ajuste de hiperparámetros (como `GridSearchCV`) vistas anteriormente.

In [237]:
#Importo todas las librerias que voy a necesitar
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
# OJO!!!! las funciones create_factor_transf_and_json, train_prepare_test_data y train_print_model que son de creacion personal estan en utils
# (archivo que ya existia en el repositorio y no toque), para mantener la legibilidad de este jupiterlab.
import utils
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif

In [238]:
# Cargamos el dataset
total_data = pd.read_csv('../data/raw/demographic_health_data.csv')
total_data.head()

,fips,TOT_POP,0-9,0-9 y/o % of total pop,19-Oct,10-19 y/o % of total pop,20-29,20-29 y/o % of total pop,30-39,30-39 y/o % of total pop,...,COPD_number,diabetes_prevalence,diabetes_Lower 95% CI,diabetes_Upper 95% CI,diabetes_number,CKD_prevalence,CKD_Lower 95% CI,CKD_Upper 95% CI,CKD_number,Urban_rural_code
0,1001,55601,6787,12.206615,7637,13.735364,6878,12.370281,7089,12.749771,...,3644,12.9,11.9,13.8,5462,3.1,2.9,3.3,1326,3
1,1003,218022,24757,11.355276,26913,12.344167,23579,10.814964,25213,11.564429,...,14692,12.0,11.0,13.1,20520,3.2,3.0,3.5,5479,4
2,1005,24881,2732,10.980266,2960,11.896628,3268,13.134520,3201,12.865239,...,2373,19.7,18.6,20.6,3870,4.5,4.2,4.8,887,6
3,1007,22400,2456,10.964286,2596,11.589286,3029,13.522321,3113,13.897321,...,1789,14.1,13.2,14.9,2511,3.3,3.1,3.6,595,2
4,1009,57840,7095,12.266598,7570,13.087828,6742,11.656293,6884,11.901798,...,4661,13.5,12.6,14.5,6017,3.4,3.2,3.7,1507,2


In [239]:
#cargamos el diccionario
df_dict = pd.read_csv('../data/raw/data_dict.csv', skip_blank_lines=True)
df_dict.head()

,Feature,Unnamed: 1,Unnamed: 2
0,fips,FIPS Code for the County,NaN
1,TOT_POP,Total Population,This data as well as all Age and Race data is ...
2,0-9,Population aged 0-9,All of the other age columns are the same but ...
3,0-9 y/o % of total pop,% of the population aged 0-9,NaN
4,10-19',NaN,NaN


In [240]:
#imprimimos el numero de filas
print(f"Numero De Filas en 'total_data': {len(total_data)}")

# Verificamos La Cantidad De Variables Por Tipos De Datos en total_data que son los datos que vamos a aplicar el EDA:
pd.Series.value_counts(total_data.dtypes)

Numero De Filas en 'total_data': 3140


float64    61
int64      45
str         2
Name: count, dtype: int64

In [241]:
print("------------------PORCENTAJE DE VALORES NULOS DEL total_data---------------------\n")
#Obtenemos que columnas tienen valores nulos y porcentage respectivamente
info_nulls = pd.DataFrame({
    '% Nullidad': total_data.isna().sum() / len(total_data) * 100
}).sort_values(by='% Nullidad', ascending=False)
info_nulls


------------------PORCENTAJE DE VALORES NULOS DEL total_data---------------------



,% Nullidad
fips,0.0
TOT_POP,0.0
0-9,0.0
0-9 y/o % of total pop,0.0
19-Oct,0.0
...,...
CKD_prevalence,0.0
CKD_Lower 95% CI,0.0
CKD_Upper 95% CI,0.0
CKD_number,0.0


In [242]:
print("------------------PORCENTAJE DE VALORES UNICOS DEL total_data---------------------\n")
#Obtenemos que columnas tienen valores unicos y porcentage respectivamente
info_uniques = pd.DataFrame({
    'Valores Únicos': total_data.nunique(),
    'Total Filas': len(total_data),
    '% Unicidad': (total_data.nunique() / len(total_data)) * 100
}).sort_values(by='% Unicidad', ascending=False)
info_uniques

------------------PORCENTAJE DE VALORES UNICOS DEL total_data---------------------



,Valores Únicos,Total Filas,% Unicidad
fips,3140,3140,100.000000
80+ y/o % of total pop,3139,3140,99.968153
70-79 y/o % of total pop,3139,3140,99.968153
60-69 y/o % of total pop,3139,3140,99.968153
% White-alone,3139,3140,99.968153
...,...,...,...
CKD_prevalence,43,3140,1.369427
CKD_Lower 95% CI,39,3140,1.242038
Active General Surgeons per 100000 Population 2018 (AAMC),32,3140,1.019108
Active Patient Care General Surgeons per 100000 Population 2018 (AAMC),30,3140,0.955414


# Analisis (Hasta Ahora):
* Tenemos un dataframe total_data con 108 columnas y 3140  filas
* 45 columnas son int64, 2 str y 61 float64
* Ninguna de esas columnas tienen valores nulos.
* No tenemos Target aun, hay que buscarlo entre tantas columnas.
* Hay muchas columnas con valores unicos por arriba del 90% como son muchas vamos a intentar quitar todas las que esten por encima de este porcentaje, ya que al ser muy unicos no nos aportar informacion relevante

In [243]:
# Parece un diccionario esta un poco desordenado y ademas el dataset tiene 108 columnas veamos que podemos hacer
# Vamos a intentar buscar palabras relacionadas con la salud como "heart,diabetes,copd,ckd,health" y vemos su definicion en el diccionario.
keywords = 'heart|diabetes|copd|ckd|health'
# Filtramos la lista de columnas que contengan esas palabras
df_salud = total_data.filter(regex=f'(?i){keywords}')
df_salud.columns
search = df_dict.iloc[df_dict["Feature"].str.contains(keywords,case=False,na=False)]
search

,Feature,Unnamed: 1,Unnamed: 2
84,Heart disease_prevalence,NaN,NaN
85,Heart disease_Lower 95% CI,NaN,NaN
86,Heart disease_Upper 95% CI,NaN,NaN
87,Heart disease_number,Population with Heart Disease,NaN
88,COPD_prevalence,NaN,NaN
89,COPD_Lower 95% CI,NaN,NaN
90,COPD_Upper 95% CI,NaN,NaN
91,COPD_number,Population with COPD,NaN
92,diabetes_prevalence,NaN,NaN
93,diabetes_Lower 95% CI,NaN,NaN


## Cual target Escojemos?:

`Heart disease_number` que es el numero de personas con enfermedades al corazon, es un target interezante y es de los pocos que el diccionario que no tiene null en su descripcion, ahora vamos a ver como limpiamos columnas, porque histogramas y graficos de cajas pueden ser mucho con tantas columnas.

In [244]:
#vamos a eliminar todas las columnas que tengan una unicidad por encima del 90% ya que tenemos muchas
# Filtramos donde el % Unicidad sea mayor a 90
columns_to_del = info_uniques[info_uniques['% Unicidad'] > 90].index.tolist()
total_data = total_data.drop(columns=columns_to_del)
total_data.head()

,0-9,19-Oct,20-29,30-39,40-49,50-59,60-69,70-79,80+,Black-alone pop,...,COPD_number,diabetes_prevalence,diabetes_Lower 95% CI,diabetes_Upper 95% CI,diabetes_number,CKD_prevalence,CKD_Lower 95% CI,CKD_Upper 95% CI,CKD_number,Urban_rural_code
0,6787,7637,6878,7089,7582,7738,5826,4050,2014,10915,...,3644,12.9,11.9,13.8,5462,3.1,2.9,3.3,1326,3
1,24757,26913,23579,25213,27338,29986,29932,20936,9368,19492,...,14692,12.0,11.0,13.1,20520,3.2,3.0,3.5,5479,4
2,2732,2960,3268,3201,3074,3278,3076,2244,1048,12042,...,2373,19.7,18.6,20.6,3870,4.5,4.2,4.8,887,6
3,2456,2596,3029,3113,3038,3115,2545,1723,785,4770,...,1789,14.1,13.2,14.9,2511,3.3,3.1,3.6,595,2
4,7095,7570,6742,6884,7474,7844,6965,4931,2335,950,...,4661,13.5,12.6,14.5,6017,3.4,3.2,3.7,1507,2


In [245]:
#ahora reducimos las columnas a 70 vamos a intentar revisar las columnas categoricas para continuar con la reduccion de datos
total_data.select_dtypes(include = ['str']).columns

Index(['COUNTY_NAME', 'STATE_NAME'], dtype='str')

In [246]:
#vemos valores unicos en las dos posibles variables catwegorizas que tenemos
STATE_NAME_unique = total_data['STATE_NAME'].unique()
COUNTY_NAME_unique = total_data['COUNTY_NAME'].unique()
print(f"Numero de STATE unicos: {len(STATE_NAME_unique)}"),
print(f"Numero de COUNTY unicos: {len(COUNTY_NAME_unique)}"),

Numero de STATE unicos: 51
Numero de COUNTY unicos: 1841


(None,)

In [247]:
#tenemos 51 valores unicos en STATE_NAME podriamos decir que es factorizable pero 1841 en COUNTY_NAME, como tenemos casi  3200 filas este ultimo casi no nos dice nada
#Eliminamos la columna COUNTY_NAME
total_data.drop(["COUNTY_NAME"],  axis = 1,  inplace = True)
# Y factorizamos STATE_NAME
utils.create_factor_transf_and_json("STATE_NAME", total_data, "SaludEjercicio3")


Json guardado en: ../data/processed/factories/SaludEjercicio3\STATE_NAME_factory_rules.json


In [ ]:
correlaciones = total_data.corr()['Heart disease_prevalence'].sort_values(ascending=False)
correlaciones = correlaciones[(correlaciones < -0.5) | (correlaciones > 0.5)]
len(correlaciones) 

28

## Que nos queda ahora?:
Ahora que hemos reducido las correlaciones 